# ⚡ Smart Grid IoT Energy Consumption & Anomaly Detection
### Course: BIS586 - Information Science & Engineering Minor Project (Section A)
### Academic Standards: IEEE 1159 (Power Quality Monitoring) & ACM SIGENERGY Guidelines

This interactive notebook demonstrates the complete exploratory data analysis, feature engineering, and statistical anomaly detection pipeline.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure local src is importable
sys.path.append(os.path.abspath('..'))
from src.data_preprocessing import load_raw_data, audit_raw_data_integrity
from src.feature_engineering import add_temporal_features, compute_dynamic_zscores
from src.anomaly_detection import detect_grid_anomalies, evaluate_anomaly_detection_performance

print("Libraries successfully imported.")

## 1. Load Raw Smart Meter Telemetry
> **Dataset Note:** This is a synthetic benchmark dataset modeled across 10 municipal zone topologies for 90 days with ground-truth IEEE 1159 electrical faults.

In [ ]:
df_raw = load_raw_data("../data/raw/raw_smart_grid_data.csv")
print(f"Loaded {len(df_raw):,} records with columns:")
print(df_raw.columns.tolist())
df_raw.head()

## 2. Execute Data Quality Audit

In [ ]:
audit_report = audit_raw_data_integrity(df_raw)
for k, v in audit_report.items():
    if k != 'missing_by_column':
        print(f"{k:<25}: {v}")

## 3. Temporal Feature Decomposition & Rolling Z-Scores
Vectorized moving average and moving standard deviation over a 24-hour diurnal window ($W=96$ steps):

In [ ]:
df_temporal = add_temporal_features(df_raw)
df_processed = compute_dynamic_zscores(df_temporal, window_size=96, z_threshold=3.0)
df_processed = detect_grid_anomalies(df_processed)
print(f"Computed {len(df_processed.columns)} features. Head preview:")
df_processed[['timestamp', 'zone_name', 'energy_consumption_kwh', 'rolling_mean_kwh', 'consumption_zscore', 'anomaly_severity']].head()

## 4. Diurnal Energy Consumption Heatmap Matrix
*Notice: This is a matrix-based visualization of mean active load indexed by Zone vs. Hour of Day, illustrating diurnal human demand patterns across urban topologies. It is not a geographic map.*

In [ ]:
pivot = df_processed.pivot_table(index='zone_name', columns='Hour', values='energy_consumption_kwh', aggfunc='mean')
plt.figure(figsize=(14, 7), dpi=150)
sns.heatmap(pivot, cmap='YlOrRd', annot=True, fmt='.1f', linewidths=0.5)
plt.title('Diurnal Energy Consumption Heatmap Matrix (Mean kWh)', fontweight='bold')
plt.xlabel('Hour of the Day (00:00 - 23:00)')
plt.ylabel('Smart Grid Zone')
plt.tight_layout()
plt.show()

## 5. Performance Evaluation Against Injected Ground Truth

In [ ]:
y_true = df_processed['is_synthetic_anomaly']
y_pred = df_processed['detected_anomaly']
metrics = evaluate_anomaly_detection_performance(y_true, y_pred)
print("=" * 50)
print(" IEEE / ACM Anomaly Detection Performance Metrics ")
print("=" * 50)
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"{k:<25}: {v*100:.2f}%")
    else:
        print(f"{k:<25}: {v:,}")